<a href="https://colab.research.google.com/github/blbl-blbl/study/blob/main/PyTorch/01_oxford_pets/02_custom_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oxford-IIIT Pet — Custom CNN

Самостоятельная CNN для 37 пород: подготовка тех же данных, устройство свёрточных блоков, формы тензоров, logits и базовая оценка модели.

> Этот notebook самодостаточен: его можно запускать сверху вниз в чистом Google Colab. Он не требует выполнения других notebook-файлов проекта.

In [ ]:
import os

# Должно быть установлено до первого использования CUDA
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import torch

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [ ]:
from torchvision.datasets import OxfordIIITPet
import matplotlib.pyplot as plt
import random

dataset = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    download=True,
)

print("Количество изображений:", len(dataset))

image, label = dataset[0]

print("Тип изображения:", type(image))
print("Размер (ширина, высота):", image.size)
print("Цветной режим:", image.mode)
print("Метка класса:", label)

indices = random.Random(42).sample(range(len(dataset)), 12)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))

for ax, index in zip(axes.flat, indices):
  image, label = dataset[index]

  ax.imshow(image)
  ax.set_title(f"Class {label} | {image.size}")
  ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

rows = []

for index in range(len(dataset)):
  image, label = dataset[index]

  size = image.size
  rows.append({
      'index': index,
      'label': label,
      'width': size[0],
      'height': size[1],
      'aspect_ratio': size[0] / size[1]
  })

metadata = pd.DataFrame(rows)


print(metadata.head())

print(
    metadata[["width", "height", "aspect_ratio"]].describe()
)

class_counts = metadata["label"].value_counts().sort_index()

print("Количество классов:", len(class_counts))
print("Минимум изображений на класс:", class_counts.min())
print("Максимум изображений на класс:", class_counts.max())

In [ ]:
from sklearn.model_selection import train_test_split

train_indices, val_indices = train_test_split(
    metadata["index"].to_numpy(),
    test_size=0.2,
    random_state=42,
    stratify=metadata["label"].to_numpy(),
)

print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Пересечение:", len(set(train_indices) & set(val_indices)))

In [ ]:
from torchvision import transforms
from torchvision.transforms import functional as TF


class ResizeWithPadding:
    def __init__(self, size=224):
        self.size = size

    def __call__(self, image):
        image = image.convert("RGB")
        width, height = image.size

        scale = self.size / max(width, height)

        new_width = max(1, round(width * scale))
        new_height = max(1, round(height * scale))

        image = TF.resize(
            image,
            [new_height, new_width],
            antialias=True,
        )

        # Берём фактические размеры после resize
        width, height = image.size

        left = (self.size - width) // 2
        right = self.size - width - left

        top = (self.size - height) // 2
        bottom = self.size - height - top

        return TF.pad(
            image,
            [left, top, right, bottom],
            fill=0,
        )

In [ ]:
basic_transform = transforms.Compose([
    ResizeWithPadding(224),
    transforms.ToTensor(),
])

image_tensor = basic_transform(image)

print("Shape:", image_tensor.shape)
print("Dtype:", image_tensor.dtype)
print("Range:", image_tensor.min().item(), image_tensor.max().item())

In [ ]:
from torch.utils.data import Subset, DataLoader

train_source = OxfordIIITPet(
    root='data',
    split='trainval',
    target_types='category',
    transform=basic_transform,
    download=False,
)

val_source = OxfordIIITPet(
    root='data',
    split='trainval',
    target_types='category',
    transform=basic_transform,
    download=False,
)

train_dataset = Subset(train_source, train_indices.tolist())
val_dataset = Subset(val_source, val_indices.tolist())

train_source.transform = basic_transform
val_source.transform = basic_transform

class_names = train_source.classes

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Classes:", len(class_names))

In [ ]:
import torch

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

images, labels = next(iter(train_loader))

print("Images:", images.shape, images.dtype)
print("Labels:", labels.shape, labels.dtype)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

### Сбор базовой CNN

Архитектура для входа `[32, 3, 224, 224]`:

| Блок | Операции | Форма выхода |
| :--- | :---- | :--- |
| 1 | `Conv2d(3, 16, 3, padding=1)` → ReLU → MaxPool2d(2) | `[32, 16, 112, 112]` |
| 2 | `Conv2d(16, 32, 3, padding=1)` → ReLU → MaxPool2d(2) | `[32, 32, 56, 56]` |
| 3 | `Conv2d(32, 64, 3, padding=1)` → ReLU → MaxPool2d(2) | `[32, 64, 28, 28]` |
| Усреднение | `AdaptiveAvgPool2d((4, 4))` | `[32, 64, 4, 4]` |
| Разворачивание | `Flatten()` | `[32, 1024]` |
| Классификатор | `Linear(1024, 37)` | `[32, 37]` |

In [ ]:
from torch import nn

class PetCNN(nn.Module):
  def __init__(self, num_classes=37):
    super().__init__()

    self.features = nn.Sequential(
        nn.Conv2d(
            in_channels=3,
            out_channels=16,
            kernel_size=3,
            padding=1
        ),
        nn.ReLU(),
        nn.MaxPool2d(2),


        nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        ),
        nn.ReLU(),
        nn.MaxPool2d(2),


        nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        ),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.AvgPool2d(
            kernel_size=7,
            stride=7
            ),
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(1024, num_classes),
    )


  def forward(self, x):
    x = self.features(x)
    logits  = self.classifier(x)
    return logits

In [ ]:
torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = PetCNN(num_classes=len(class_names)).to(device)

images, labels = next(iter(val_loader))

model.eval()

with torch.inference_mode():
  logits = model(images.to(device))

  loss = nn.CrossEntropyLoss()(
      logits,
      labels.to(device)
  )

print("Logits shape:", logits.shape)
print("Loss before training:", loss.item())

parameter_count = sum(p.numel() for p in model.parameters())
print("Parameters:", parameter_count)

### Оценка качества для первого эксперимента

Выбранные метрики:

| **Метрика** | **Что измеряет** |
| :--- | :--- |
| Accuracy | Долю фотографий, для которых первый выборанный класс правильный |
| Macro-F1 | Среднее F1 по всем 37 породам с одинаковым весом каждой породы |
| Top-3 accuracy | Долю фотографий, где правильная порода попала в тройку выбранных |


## Функция валидации

In [ ]:
import torch
from sklearn.metrics import f1_score


def evaluate_metrics(model, dataloader, loss_fn, device, num_classes):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_top3_correct = 0
    total_objects = 0

    all_labels = []
    all_predictions = []

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = loss_fn(logits, labels)

            predictions = logits.argmax(dim=1)
            top3 = logits.topk(k=3, dim=1).indices

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (predictions == labels).sum().item()
            total_top3_correct += (
                (top3 == labels.unsqueeze(1))
                .any(dim=1)
                .sum()
                .item()
            )
            total_objects += batch_size

            all_labels.extend(labels.cpu().tolist())
            all_predictions.extend(predictions.cpu().tolist())

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        labels=list(range(num_classes)),
        average="macro",
        zero_division=0,
    )

    return {
        "loss": total_loss / total_objects,
        "accuracy": total_correct / total_objects,
        "macro_f1": macro_f1,
        "top3_accuracy": total_top3_correct / total_objects,
    }

In [ ]:
metrics = evaluate_metrics(
    model,
    val_loader,
    nn.CrossEntropyLoss(),
    device,
    num_classes=len(class_names)
)

print(metrics)
print(f"Macro-F1: {metrics['macro_f1']:.4f}")